In [1]:
import pandas as pd
import numpy as np
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🖥️ Running on device: {device}")

🖥️ Running on device: cpu


 Data Preprocessing and Tensor Conversion.

In [ ]:
df = pd.read_csv("../data/tickets.csv")

X_train_text, X_test_text, y_train_text, y_test_text = train_test_split(
    df["ticket_text"], df["category"],
    test_size=0.2, random_state=42, stratify=df["category"],
)

vectorizer = TfidfVectorizer(lowercase=True, stop_words="english", max_features=500)
X_train = vectorizer.fit_transform(X_train_text).toarray().astype("float32")
X_test  = vectorizer.transform(X_test_text).toarray().astype("float32")

X_train = torch.tensor(X_train, dtype=torch.float32)
X_test  = torch.tensor(X_test,  dtype=torch.float32)


le = LabelEncoder()
y_train_encoded = le.fit_transform(y_train_text)
y_test_encoded  = le.transform(y_test_text)

y_train = torch.tensor(y_train_encoded, dtype=torch.long)
y_test  = torch.tensor(y_test_encoded,  dtype=torch.long)


In [4]:
classes, counts = np.unique(y_train_encoded, return_counts=True)
weights = 1.0 / counts
weights = weights / weights.sum()  # Normalize
class_weights_tensor = torch.tensor(weights, dtype=torch.float32).to(device)

 2. Production Habit: Handle Deep Learning Class Imbalance

In [5]:
# --- INITIAL DATA & WEIGHTS REPORT ---
print("\n Initial Data Distribution & Class Weights Report:")
print("-" * 55)
for cls_idx, count, weight in zip(classes, counts, weights):
    class_name = le.classes_[cls_idx]
    print(f"Class {cls_idx} ({class_name:<10}) | Samples: {count:<3} | Weight Multiplier: {weight:.4f}")
print("-" * 55)
print(f"Final Class Weights Tensor (on {device}):\n{class_weights_tensor}\n")


 Initial Data Distribution & Class Weights Report:
-------------------------------------------------------
Class 0 (billing   ) | Samples: 32  | Weight Multiplier: 0.2500
Class 1 (refund    ) | Samples: 32  | Weight Multiplier: 0.2500
Class 2 (shipping  ) | Samples: 32  | Weight Multiplier: 0.2500
Class 3 (technical ) | Samples: 32  | Weight Multiplier: 0.2500
-------------------------------------------------------
Final Class Weights Tensor (on cpu):
tensor([0.2500, 0.2500, 0.2500, 0.2500])



# 3. Dataset + DataLoader

In [6]:
class TicketDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_loader = DataLoader(TicketDataset(X_train, y_train), batch_size=16, shuffle=True)
test_loader  = DataLoader(TicketDataset(X_test,  y_test),  batch_size=16, shuffle=False)


4. Model Architecture & Setup

In [7]:
n_features = X_train.shape[1]     
n_classes  = len(le.classes_)     

model = nn.Sequential(
    nn.Linear(n_features, 64),
    nn.ReLU(),
    nn.Dropout(0.3),
    nn.Linear(64, n_classes),
).to(device)  # Move model weights to the target device (GPU/CPU)

# Pass the calculated class weights to penalize minority misclassifications
loss_fn = nn.CrossEntropyLoss(weight=class_weights_tensor)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4) # Added L2 Regularization


# ---------------------------------------------------------------
# 5. Production Training Loop with Val Tracking
# ---------------------------------------------------------------

In [8]:

for epoch in range(30):
    model.train()
    epoch_loss = 0.0
    for xb, yb in train_loader:
        # Move batch data to the active device
        xb, yb = xb.to(device), yb.to(device)
        
        logits = model(xb)
        loss = loss_fn(logits, yb)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * len(xb)
        
    # Validation Tracking: Check how the test set is doing concurrently
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for val_xb, val_yb in test_loader:
            val_xb, val_yb = val_xb.to(device), val_yb.to(device)
            val_logits = model(val_xb)
            val_loss += loss_fn(val_logits, val_yb).item() * len(val_xb)

    print(f"Epoch {epoch+1:2d} | Train Loss: {epoch_loss / len(X_train):.4f} | Val Loss: {val_loss / len(X_test):.4f}")

# ---------------------------------------------------------------
# 6. Evaluation
# ---------------------------------------------------------------
model.eval()
with torch.no_grad():
   
    logits = model(X_test.to(device))
    preds = logits.argmax(dim=1).cpu().numpy()  
print("\n Production PyTorch Evaluation Report:")
print(classification_report(y_test.numpy(), preds, target_names=le.classes_, zero_division=0))
print("\n Confusion Matrix:")
print(confusion_matrix(y_test.numpy(), preds))

Epoch  1 | Train Loss: 1.3862 | Val Loss: 1.3842
Epoch  2 | Train Loss: 1.3754 | Val Loss: 1.3780
Epoch  3 | Train Loss: 1.3667 | Val Loss: 1.3709
Epoch  4 | Train Loss: 1.3542 | Val Loss: 1.3627
Epoch  5 | Train Loss: 1.3375 | Val Loss: 1.3525
Epoch  6 | Train Loss: 1.3230 | Val Loss: 1.3399
Epoch  7 | Train Loss: 1.3015 | Val Loss: 1.3247
Epoch  8 | Train Loss: 1.2717 | Val Loss: 1.3064
Epoch  9 | Train Loss: 1.2454 | Val Loss: 1.2851
Epoch 10 | Train Loss: 1.2027 | Val Loss: 1.2600
Epoch 11 | Train Loss: 1.1571 | Val Loss: 1.2300
Epoch 12 | Train Loss: 1.0989 | Val Loss: 1.1955
Epoch 13 | Train Loss: 1.0508 | Val Loss: 1.1580
Epoch 14 | Train Loss: 0.9862 | Val Loss: 1.1194
Epoch 15 | Train Loss: 0.9240 | Val Loss: 1.0779
Epoch 16 | Train Loss: 0.8540 | Val Loss: 1.0351
Epoch 17 | Train Loss: 0.7956 | Val Loss: 0.9934
Epoch 18 | Train Loss: 0.7248 | Val Loss: 0.9500
Epoch 19 | Train Loss: 0.6660 | Val Loss: 0.9097
Epoch 20 | Train Loss: 0.5998 | Val Loss: 0.8689
Epoch 21 | Train Los

In [9]:
import pickle

print("\n Archiving production assets for deployment...")

# 1. Save the PyTorch Model Weights
torch.save(model.state_dict(), "ticket_nn_model.pt")

# 2. Save the Sklearn Vectorizer and Label Encoder using pickle
with open("tfidf_vectorizer.pkl", "wb") as f:
    pickle.dump(vectorizer, f)

with open("label_encoder.pkl", "wb") as f:
    pickle.dump(le, f)

print(" All assets saved successfully: 'ticket_nn_model.pt', 'tfidf_vectorizer.pkl', 'label_encoder.pkl'")



 Archiving production assets for deployment...
 All assets saved successfully: 'ticket_nn_model.pt', 'tfidf_vectorizer.pkl', 'label_encoder.pkl'
